# Export results → `results/boolean/` (no GPU)

**Runtime → CPU is fine. Runtime → Run all.** Nothing here re-runs probing or
the causal sweep — it reads the runs already sitting in your Drive and writes
the reviewable part into the repo.

| cell | does | cost |
|---|---|---|
| 1 | restore every language's runs from Drive | ~1 min |
| 2 | export **probe** summaries — all languages, one pass | seconds |
| 3 | regenerate probe **figures** — one pass per language found | ~2 min each |
| 4 | export **causal** summaries + layer profiles — all languages | seconds |
| 5 | show what is about to be committed | — |
| 6 | commit and push | — |

Cell 3 is the only per-language part, because `make_figures.py` takes a
`--lang` flag. Everything else globs across all languages at once.

If you only want the numbers in the repo and not the PNGs, run **1, 2, 4, 5, 6**
and skip 3 — that is about two minutes total.


In [ ]:
# 1 - setup + restore from Drive. No torch, no transformers: nothing here
#     loads a model, so the install is small and the runtime can be CPU.
import os, pathlib, glob, re
REPO = "/content/code-model-interpretability"
if not pathlib.Path(REPO).exists():
    !git clone -q https://github.com/nolanlwin/code-model-interpretability.git {REPO}
%cd {REPO}
!git fetch -q origin && git checkout -q -B main origin/main && git pull -q
!git log --oneline -1
!pip install -q numpy scikit-learn matplotlib

from google.colab import drive
drive.mount("/content/drive")
PROBE_DEST  = "/content/drive/MyDrive/code-model-interpretability/xlcost"
CAUSAL_DEST = "/content/drive/MyDrive/code-model-interpretability/causal"
!mkdir -p outputs/probe_results outputs/causal outputs/xlcost_occ data/xlcost results/boolean
!cp -n {PROBE_DEST}/probe_results/* outputs/probe_results/ 2>/dev/null || true
!cp -n {PROBE_DEST}/xlcost_occ/*    outputs/xlcost_occ/    2>/dev/null || true
!cp -n {PROBE_DEST}/data_xlcost/*   data/xlcost/           2>/dev/null || true
!cp -n {CAUSAL_DEST}/*.json         outputs/causal/        2>/dev/null || true

probe_files  = glob.glob("outputs/probe_results/*_problem.json")
causal_files = glob.glob("outputs/causal/*.json")
LANGS = sorted({m.group(1) for f in probe_files
                if (m := re.match(r".*/(\w+?)_(train|valid|test)_\w+_problem\.json$", f))})
print(f"\nrestored {len(probe_files)} probe runs, {len(causal_files)} causal runs")
print(f"languages found: {LANGS or 'NONE - check the Drive paths above'}")
if not probe_files and not causal_files:
    raise SystemExit("Nothing restored. Your runs may be under a different Drive "
                     "path - list /content/drive/MyDrive/code-model-interpretability and adjust "
                     "PROBE_DEST / CAUSAL_DEST above.")


In [ ]:
# 2 - probe summaries for EVERY language in one pass.
#     Each row carries rho, computed from that run's own test predictions:
#     the macro-F1 movement caused by one test occurrence of the smallest
#     class flipping. Margins smaller than rho are flagged, because that is
#     the fact that rewrote section 4.4.
!python scripts/export_probe.py --in outputs/probe_results --out results/boolean/probe
print()
print(open("results/boolean/probe/SUMMARY.md").read())


In [ ]:
# 3 - probe figures, one pass per language (make_figures.py is language-scoped).
#     ~2 min each, CPU. Skip this cell if you only want the numbers.
#     Baselines are refreshed first so they include every baseline the current
#     code defines and emit the predictions the probe-vs-baseline CI needs.
SPLIT = "train"
for lang in LANGS:
    print(f"\n######## {lang} ########")
    for probe in sorted(glob.glob(f"outputs/probe_results/{lang}_{SPLIT}_*_problem.json")):
        m = re.match(rf".*{lang}_{SPLIT}_(?!C\d)(\w+)_problem\.json$", probe)
        if not m:
            continue
        sample = probe + ".sample_ids.json"
        if not os.path.exists(sample):
            print(f"  skip {m.group(1)}: no sample_ids file"); continue
        !python scripts/baselines.py run \
          --occurrences outputs/xlcost_occ/{lang}_{SPLIT}.jsonl \
          --canonical data/xlcost/{lang}_{SPLIT}.jsonl \
          --sample-ids {sample} --split-policy repo \
          --output outputs/probe_results/{lang}_{SPLIT}_{m.group(1)}_baselines_capped.json
    !python scripts/make_figures.py --results-dir outputs/probe_results \
      --lang {lang} --split {SPLIT} --out results/boolean \
      --reference-delta -0.272 \
      --reference-label "index role, Python/train, Qwen2.5-1.5B"

# re-export so the summaries pick up the refreshed baselines
!python scripts/export_probe.py --in outputs/probe_results --out results/boolean/probe


In [ ]:
# 4 - causal summaries + layer-profile figures for EVERY language, one pass.
!python scripts/export_causal.py --in outputs/causal --out results/boolean/causal --role boolean
print()
print(open("results/boolean/causal/SUMMARY.md").read())


In [ ]:
# 5 - what is about to be committed
from IPython.display import Image, display
import glob, os
total = 0
for f in sorted(glob.glob("results/boolean/**/*", recursive=True)):
    if os.path.isfile(f):
        kb = os.path.getsize(f) / 1024
        total += kb
        print(f"  {kb:8.1f} KB  {f}")
print(f"\n  {total/1024:.2f} MB total")
for f in sorted(glob.glob("results/boolean/**/*.png", recursive=True))[:6]:
    print(f); display(Image(f))


In [ ]:
# 6 - commit and push.
#     Needs a Colab secret GH_TOKEN: a FINE-GRAINED personal access token,
#     this repository only, Contents = Read and write. Not a GPG key -- a GPG
#     key signs commits, it cannot authenticate a push.
#     No token? Download results/boolean/ from the file browser on the left
#     and commit from your local clone. Same result, nothing to revoke later.
from google.colab import userdata
import os
try:
    tok = userdata.get("GH_TOKEN")
except Exception:
    tok = None
if not tok:
    print("No GH_TOKEN secret - download results/boolean/ and commit locally.")
else:
    os.environ["GH_TOKEN"] = tok
    !git config user.email "naingoolwin.astrio@gmail.com"
    !git config user.name "naingoolwin"
    !git add results/boolean
    !git commit -q -m "Probe and causal results for boolean, all languages" || echo "nothing to commit"
    !git push -q https://$GH_TOKEN@github.com/nolanlwin/code-model-interpretability.git HEAD:main && echo "pushed - see github.com/nolanlwin/code-model-interpretability/tree/main/results/boolean"
